In [ ]:
import json
import sys
sys.path.append('/home/wwd/codebox/ODT-main')

def getfolder(target_path, timepoint):
    # 构建目标路径
    file_path = os.path.join(target_path, timepoint)
    # 获取该路径下的所有文件夹
    folders = [f for f in os.listdir(file_path) if os.path.isdir(os.path.join(file_path, f))]
    # 筛选出以 'checkpoints-' 开头的文件夹，并提取其中的数字
    checkpoint_folders = []
    for folder in folders:
        if folder.startswith('checkpoint-'):
            # 提取数字部分并转换为整数
            try:
                num = int(folder.split('-')[1])
                checkpoint_folders.append((num, folder))
            except ValueError:
                continue  # 如果文件夹名后面的数字不合法，则跳过

    # 找到数字最大的一项
    if checkpoint_folders:
        max_num, max_folder = max(checkpoint_folders, key=lambda x: x[0])
        return os.path.join(target_path, timepoint, max_folder)
    else:
        print("No checkpoints found.")
def getValues(target_path, type='spatial'):
    # 获取历史accuracy和f1分数
    timepoints = ['0h', '12h', '1.5d', '3d', '5d', '10d', 'WT']
    history_best = {}
    if type == 'spatial':
        for timepoint in timepoints:
            history_best[timepoint] = []
            history_item = []
            file_path = target_path + timepoint + '/history.txt'
            with open(file_path, 'r') as file:
                lines = file.readlines()
            for line in lines:
                line = line.strip('\n')
                line = line.split(' ')
                history_item.append([float(line[0]), float(line[1])])
            sorted_history_item = sorted(history_item, key=lambda x: x[0], reverse=True)
            history_best[timepoint] = sorted_history_item[0]

    if type == 'gene':
        for timepoint in timepoints:
            history_best[timepoint] = []
            history_item = []
            file_path = getfolder(target_path, timepoint) + '/trainer_state.json'
            # 读取并解析 JSON 文件
            with open(file_path, 'r') as file:
                data = json.load(file)
            history_item = []
            for item in data['log_history']:
                if 'eval_accuracy' in item and 'eval_macro_f1' in item:
                    history_item.append([
                        item['eval_accuracy'],
                        item['eval_macro_f1']
                    ])
            sorted_history_item = sorted(history_item, key=lambda x: x[0], reverse=True)
            history_best[timepoint] = sorted_history_item[0]
    return history_best


In [ ]:
getValues(target_path ='../result/model_results/geneformer/',type = 'gene')
getValues(target_path ='../result/model_results/gene_raw/',type = 'gene')
getValues(target_path ='../result/model_results/spatial_raw/')

In [ ]:
import umap
from sklearn.metrics import silhouette_score
from collections import defaultdict, Counter
import numpy as np
from sklearn.cluster import KMeans
from matplotlib import pyplot as plt
import scanpy as sc
import datasets
from sklearn.decomposition import IncrementalPCA, PCA
import random
from tqdm import tqdm
import pickle
import os
import torch

Genes_all = ['SMESG000019132','SMESG000081615','SMESG000075225','SMESG000010938','SMESG000077295','SMESG000021611']
Genes_all2 = ['SMESG000005389', 'SMESG000046293', 'SMESG000063577', 'SMESG000020104', 'SMESG000060238', 'SMESG000029273', 'SMESG000053725', 'SMESG000070594', 'SMESG000070103', 'SMESG000049472', 'SMESG000016858', 'SMESG000060978', 'SMESG000037081', 'SMESG000016751', 'SMESG000056159', 'SMESG000069029', 'SMESG000045786', 'SMESG000070150', 'SMESG000064333', 'SMESG000025543', 'SMESG000048533', 'SMESG000060148', 'SMESG000079942', 'SMESG000020739', 'SMESG000042345', 'SMESG000057772', 'SMESG000033647', 'SMESG000048749', 'SMESG000036986', 'SMESG000060356', 'SMESG000030852', 'SMESG000044282', 'SMESG000025390', 'SMESG000061064', 'SMESG000074748', 'SMESG000078501', 'SMESG000042131', 'SMESG000051493', 'SMESG000049972', 'SMESG000074761', 'SMESG000072824']
gene_eyes = ['SMESG000081129','SMESG000025308','SMESG000025307']
# with open('./datasets_3D/all_timepoints.pkl', 'rb') as file:
with open('../dataSets/datasets_3D/cell_3D_all.pkl', 'rb')as file:
    dataset_3D = pickle.load(file)
    print(len(dataset_3D))
# #去除低表达的细胞
dataset_3D = [item for item in dataset_3D if(item['length']>=50)]

def makeModelEmbeddings():
    # 用训练好的模型获取向量
    timepoints = ['0hpa1', '12hpa1', '36hpa1', '3dpa1', '5dpa1', '10dpa1', 'WT']
    timepoints2 = ['0h', '12h','1.5d', '3d', '5d', '10d', 'WT']
    # 模型只有这几个
    for timepoint in timepoints:
        # 读取模型
        timepoint2 = timepoints2[timepoints.index(timepoint)]
        model_path = '../models/time_spatial/' +timepoint2+ "/bert_with_spatial_model_best.pth"
        spatial_model = torch.load(model_path)
        spatial_model = spatial_model.to('cuda')
        index = 0
        # print(spatial_model)
        filename = os.path.join("../3D_datasetSplit", f"3Ddataset_timepoint_{timepoint}.pkl")
        with open(filename,'rb')as file:
            data = pickle.load(file)
        with open('../dataSets/datasets_3D/pca_results_dict_'+timepoint+'_2.pkl', 'rb') as file:
            data_pca = pickle.load(file)
        spatial_model.eval()
        for cell in tqdm(data):
            # 1. 构造 input_ids（加 batch 维）
            input_ids = cell['input_ids'][:2048]
            random.shuffle(input_ids)
            input_ids = torch.tensor([input_ids])  # shape: [1, seq_len]
            input_ids = input_ids.long().to('cuda')
            # 2. 构造 lengths（加 batch 维）
            lengths = torch.tensor([[cell['length']]]).to('cuda')  # shape: [1, 1]
            # 3. 构造 spatial 信息（取 x, y，shape: [1, 2]）
            spatial = torch.tensor([cell['pos'][:2]]).float().to('cuda')
            # 4. 注意力掩码（shape: [1, seq_len]）
            attention_mask = (input_ids != 0).long().to('cuda')
            # 5. 前向传播，调用模型
            with torch.no_grad():
                _, embeddings, _, _ = spatial_model(
                    epoch=1,
                    input_ids=input_ids,
                    spatial_info=spatial,
                    attention_mask=attention_mask,
                    gene_length=lengths
                )
            cell['model_embeddings'] = embeddings[0]
            if(cell['cell_name'] in data_pca):
                cell['pca_embeddings'] = data_pca[cell['cell_name']]

            # 找到对应的pca向量
        target_name = os.path.join("./3D_datasetSplit", f"3Ddataset_timepoint_{timepoint}_emb.pkl")
        with open(target_name,'wb')as file:
            pickle.dump(data,file)
        print(timepoint +'end ')

        pass
    pass

def makeGeneFile(gene):
    with open('../dataSets/gene_tk.pkl','rb')as f:
        token_dictionary = pickle.load(f)
    # 先转换token_id
    gene_tk_id = token_dictionary[gene]
    # 创建对应的文件夹
    file_path = './3D_makeGeneVisualization/'+gene
    try:
        os.makedirs(file_path, exist_ok=True)  # exist_ok=True 表示如果文件夹已存在，不会引发错误
        print(f"文件夹 '{file_path}' 创建成功。")
    except Exception as e:
        print(f"创建文件夹时发生错误: {e}")
    all_cell_pos = {}
    for cell in tqdm(dataset_3D):
        time_point = cell['timepoint']
        input_ids = cell['input_ids']
        if(gene_tk_id in input_ids):
            if(time_point in all_cell_pos):
                all_cell_pos[time_point].append(cell['pos2'])
            else:
                all_cell_pos[time_point] = [cell['pos2']]
    for time_point in all_cell_pos:
        with open(file_path+'/'+time_point+'.txt', 'w') as file:
            for pos in all_cell_pos[time_point]:
                file.write(str(pos[0]))
                file.write(' ')
                file.write(str(pos[1]))
                file.write(' ')
                file.write(str(pos[2] ))
                file.write(' ')
                file.write('1')
                file.write('\n')
    print(gene + ' end')
def makeGeneFiles1():
    # 所有细胞的空间位置
    file_path = './3D_makeGeneVisualization/'+'All'
    try:
        os.makedirs(file_path, exist_ok=True)  # exist_ok=True 表示如果文件夹已存在，不会引发错误
        print(f"文件夹 '{file_path}' 创建成功。")
    except Exception as e:
        print(f"创建文件夹时发生错误: {e}")

    all_cell_pos = {}
    for cell in tqdm(dataset_3D):
        time_point = cell['timepoint']
        if(time_point in all_cell_pos):
            all_cell_pos[time_point].append(cell['pos2'])
        else:
            all_cell_pos[time_point] = [cell['pos2']]
    for time_point in all_cell_pos:
        with open(file_path+'/'+time_point+'.txt', 'w') as file:
            for pos in all_cell_pos[time_point]:
                file.write(str(pos[0]))
                file.write(' ')
                file.write(str(pos[1]))
                file.write(' ')
                file.write(str(pos[2]))
                file.write(' ')
                file.write('1')
                file.write('\n')
    for gene in Genes_all:
        makeGeneFile(gene)
    for gene in Genes_all2:
        makeGeneFile(gene)
    for gene in gene_eyes:
        makeGeneFile(gene)

def makeGeneFile2(gene):
    # 先转换token_id
    with open('../dataSets/gene_tk.pkl','rb')as file:
        token_dictionary = pickle.load(file)
    gene_tk_id = token_dictionary[gene]
    all_cell_pos = {}
    for cell in tqdm(dataset_3D):
        time_point = cell['timepoint']
        input_ids = cell['input_ids']
        if(gene_tk_id in input_ids):
            if(time_point in all_cell_pos):
                all_cell_pos[time_point].append(cell['pos2'])
            else:
                all_cell_pos[time_point] = [cell['pos2']]
    for time_point in all_cell_pos:
        with open('./3D_makeGeneVisualization2/'+time_point+'/'+gene+'.txt', 'w') as file:
            for pos in all_cell_pos[time_point]:
                file.write(str(pos[0]))
                file.write(' ')
                file.write(str(pos[1]))
                file.write(' ')
                file.write(str(pos[2]))
                file.write(' ')
                file.write('1')
                file.write('\n')
    print(gene + ' end')
def makeGeneFiles2():
    # 所有细胞的空间位置
    timepoints = ['0hpa1','12hpa1','36hpa1','3dpa1','5dpa1','7dpa1','10dpa1','14dpa1','WT']
    for timepoint in timepoints:
        file_path = './3D_makeGeneVisualization2/' + timepoint
        try:
            os.makedirs(file_path, exist_ok=True)  # exist_ok=True 表示如果文件夹已存在，不会引发错误
            print(f"文件夹 '{file_path}' 创建成功。")
        except Exception as e:
            print(f"创建文件夹时发生错误: {e}")
    all_cell_pos = {}
    for cell in tqdm(dataset_3D):
        time_point = cell['timepoint']
        if (time_point in all_cell_pos):
            all_cell_pos[time_point].append(cell['pos2'])
        else:
            all_cell_pos[time_point] = [cell['pos2']]
    for time_point in all_cell_pos:
        with open('./3D_makeGeneVisualization2/' + time_point + '/All.txt', 'w') as file:
            for pos in all_cell_pos[time_point]:
                file.write(str(pos[0]))
                file.write(' ')
                file.write(str(pos[1]))
                file.write(' ')
                file.write(str(pos[2] ))
                file.write(' ')
                file.write('0.1')
                file.write('\n')
    for gene in Genes_all:
        makeGeneFile2(gene)
    for gene in Genes_all2:
        makeGeneFile2(gene)
    for gene in gene_eyes:
        makeGeneFile2(gene)
    pass
# makeGeneFiles2()
def genePcaModel():
    # 时间点
    timepoints = ['0hpa1', '12hpa1', '36hpa1', '3dpa1', '5dpa1', '7dpa1', '10dpa1', '14dpa1', 'WT']
    # 遍历所有时间点的数据,确定统一的基因列表
    all_genes = set()
    for timepoint in timepoints:
        print(f"Processing timepoint: {timepoint}")
        # 加载 h5ad 数据
        adata = sc.read_h5ad('/home/wwd/codebox/BIoInformation/planrian_3ds/raw_Data/' + timepoint + '.h5ad')
        # 提取当前timepoint的基因列表，并将其加入到all_genes中
        # 清理基因名称，去除空格并统一大小写
        current_genes = [gene.strip().lower() for gene in adata.var_names.tolist()]
        all_genes.update(current_genes)
    # 将所有基因合集转换为列表，并按字母顺序排序
    unified_genes = sorted(list(all_genes))
    with open('./model_3D/unified_genes.pkl', 'wb') as f:
        pickle.dump(unified_genes,f)
    print(len(unified_genes))
    print('begin pca')
    # 设置降维目标维度
    n_components = 256
    batch_size = 1000
    # 初始化 IncrementalPCA，用于处理大规模数据
    pca_model = IncrementalPCA(n_components=n_components)
    for timepoint in timepoints:
        # 加载数据集和单细胞基因表达
        print(timepoint)
        this_Dataset = datasets.Dataset.load_from_disk(
            '/home/wwd/codebox/BIoInformation/planrian_3ds/datasets/' + timepoint + '_raw.datasets')
        cell_names = this_Dataset['cell_name']
        # 加载 h5ad 数据
        adata = sc.read_h5ad('/home/wwd/codebox/BIoInformation/planrian_3ds/raw_Data/' + timepoint + '.h5ad')
        # 确保cell_names与adata.obs_names对应
        # 获取当前时间点中基因的索引
        # 清理基因名称，去除空格并统一大小写
        current_genes = [gene.strip().lower() for gene in adata.var_names.tolist()]
        gene_index_map = {gene: idx for idx, gene in enumerate(current_genes)}
        # 遍历每个细胞，提取表达数据
        cell_expressions = []
        for j_id in tqdm(range(len(cell_names))):
            cell_name = cell_names[j_id]
            # 提取该细胞的基因表达
            cell_expression_raw = adata[cell_name, :].X.toarray().flatten()  # 获取该细胞的基因表达数据并展平为一维数组
            cell_expression = np.zeros(len(unified_genes))
            for j, gene in enumerate(unified_genes):
                if gene in gene_index_map:
                    gene_idx = gene_index_map[gene]
                    cell_expression[j] = cell_expression_raw[gene_idx]
            cell_expressions.append(cell_expression)
            # 每批次训练一次PCA模型
            if (j_id + 1) % batch_size == 0:
                try:
                    print(f"{(j_id + 1) // batch_size} PCA model training")
                    all_cells_expression = np.array(cell_expressions)
                    all_cells_expression = all_cells_expression.astype(np.float32)  # 确保数据格式正确
                    pca_model.partial_fit(all_cells_expression)  # 使用partial_fit进行训练
                except Exception as e:
                    print(f"Error during PCA update: {e}, current batch size: {len(cell_expressions)}")
                # 重置cell_expressions为下一批次
                cell_expressions = []
    with open('./model_3D/all_cells_pca_model.pkl', 'wb') as f:
        pickle.dump(pca_model,f)
    print('pca saved')
    pass

def genePcaEmbeddings(model_path = '../models/model_3D/all_cells_pca_model.pkl'):
    with open(model_path, 'rb') as f:
        pca_model = pickle.load(f)
    print(pca_model)
    timepoints = ['0hpa1', '12hpa1', '36hpa1', '3dpa1', '5dpa1', '7dpa1', '10dpa1', '14dpa1', 'WT']
    with open('../models/model_3D/unified_genes.pkl', 'rb') as f:
        unified_genes = pickle.load(f)
    batch_size = 10000
    # 训练完成后，进行降维
    for timepoint in timepoints:
        # 加载数据集和单细胞基因表达
        this_Dataset = datasets.Dataset.load_from_disk(
            '/home/wwd/codebox/BIoInformation/planrian_3ds/datasets/' + timepoint + '_raw.datasets')
        cell_names = this_Dataset['cell_name']
        # 加载 h5ad 数据
        adata = sc.read_h5ad('/home/wwd/codebox/BIoInformation/planrian_3ds/raw_Data/' + timepoint + '.h5ad')
        # 清理基因名称，去除空格并统一大小写
        current_genes = [gene.strip().lower() for gene in adata.var_names.tolist()]
        gene_index_map = {gene: idx for idx, gene in enumerate(current_genes)}
        # 遍历每个细胞，提取表达数据
        cell_names2 = []
        cell_expressions = []
        pca_dict = {}
        for j_id in tqdm(range(len(cell_names))):
            cell_name = cell_names[j_id]
            cell_names2.append(cell_name)
            # 提取该细胞的基因表达
            cell_expression_raw = adata[cell_name, :].X.toarray().flatten()  # 获取该细胞的基因表达数据并展平为一维数组
            cell_expression = np.zeros(len(unified_genes))
            for j, gene in enumerate(unified_genes):
                if gene in gene_index_map:
                    gene_idx = gene_index_map[gene]
                    cell_expression[j] = cell_expression_raw[gene_idx]
            cell_expressions.append(cell_expression)
            if (j_id + 1) % batch_size == 0:
                print(str((j_id+1) // batch_size)+' start pca making')
                all_cells_expression = np.array(cell_expressions)
                # 将数据类型从 float64 转换为 float32
                all_cells_expression = all_cells_expression.astype(np.float32)
                cell_embeddings = pca_model.transform(all_cells_expression)
                for id in range(len(cell_names2)):
                    cell_name = cell_names2[id]
                    pca_dict[cell_name] = cell_embeddings[id]
                print(str((j_id+1) // batch_size)+' pca end')
                cell_expressions = []
                cell_names2 = []
        with open('./3d_test/pca_results_dict_'+timepoint+'_2.pkl', 'wb') as f:
            pickle.dump(pca_dict, f)
        print(timepoint +' end')
    try:
        this_Dataset = datasets.Dataset.load_from_disk(
            '/home/wwd/codebox/BIoInformation/planrian_3ds/datasets/' + timepoint + '_raw.datasets')
        cell_names = this_Dataset['cell_name']
        # 加载 h5ad 数据
        adata = sc.read_h5ad('/home/wwd/codebox/BIoInformation/planrian_3ds/raw_Data/' + timepoint + '.h5ad')
        # 清理基因名称，去除空格并统一大小写
        current_genes = [gene.strip().lower() for gene in adata.var_names.tolist()]
        gene_index_map = {gene: idx for idx, gene in enumerate(current_genes)}
        # 遍历每个细胞，提取表达数据
        cell_names2 = []
        cell_expressions = []
        umap_dict = {}
        for j_id in tqdm(range(len(cell_names))):
            cell_name = cell_names[j_id]
            cell_names2.append(cell_name)
        # 提取该细胞的基因表达
        cell_expression_raw = adata[cell_name, :].X.toarray().flatten()  # 获取该细胞的基因表达数据并展平为一维数组
        cell_expression = np.zeros(len(unified_genes))
        for j, gene in enumerate(unified_genes):
            if gene in gene_index_map:
                gene_idx = gene_index_map[gene]
                cell_expression[j] = cell_expression_raw[gene_idx]
        cell_expressions.append(cell_expression)
        all_cells_expression = np.array(cell_expressions)
        # 将数据类型从 float64 转换为 float32
        all_cells_expression = all_cells_expression.astype(np.float32)
        # 实例化 UMAP 模型
        reducer = umap.UMAP(n_components=64, random_state=42)
        # 执行降维
        umap_embeddings = reducer.fit_transform(all_cells_expression)
        for id in range(len(cell_names2)):
            cell_name = cell_names2[id]
            umap_dict[cell_name] = {'umap_embeddings':umap_embeddings[id]}
        with open('./3d_test/pca_results_dict_' + 'WT' + '.pkl', 'wb') as f:
            pickle.dump(pca_dict, f)
    except:
        print('WT umap err')


def Cluster_make():
    timepoints = ['0hpa1', '12hpa1', '36hpa1', '3dpa1', '5dpa1', '10dpa1', 'WT']
    for timepoint in timepoints:
        target_name = os.path.join("../3D_datasetSplit", f"3Ddataset_timepoint_{timepoint}_emb.pkl")
        with open(target_name, 'rb') as file:
            cell_list = pickle.load(file)
        new_cell_list = []
        try:
            # 先筛选出同时拥有 'pca_embeddings' 和 'model_embeddings' 的 cell
            valid_cells = [cell for cell in cell_list if 'pca_embeddings' in cell and 'model_embeddings' in cell]
            if len(valid_cells) == 0:
                print(f"{timepoint} 没有足够的有效细胞用于聚类。")
            else:
                # PCA 聚类
                pca_embeddings = np.stack([cell['pca_embeddings'][:64] for cell in valid_cells])
                kmeans = KMeans(n_clusters=10, random_state=0, max_iter=1000)
                pca_labels = kmeans.fit_predict(pca_embeddings)
                # model_embeddings 聚类
                model_embeddings = np.stack([cell['model_embeddings'][:-1].cpu().numpy() for cell in valid_cells])
                pca_model_low = PCA(n_components=64)
                model_reduced = pca_model_low.fit_transform(model_embeddings)
                kmeans = KMeans(n_clusters=10, random_state=0, max_iter=1000)
                model_labels = kmeans.fit_predict(model_reduced)

                # umap 聚类（仅非 WT 且存在 umap_embeddings）
                if timepoint != 'WT':
                    umap_cells = [cell for cell in valid_cells if 'umap_embeddings' in cell]
                    if len(umap_cells) > 0:
                        umap_embeddings = np.stack([cell['umap_embeddings'] for cell in umap_cells])
                        kmeans = KMeans(n_clusters=10, random_state=0)
                        umap_labels = kmeans.fit_predict(umap_embeddings)
                        umap_label_dict = {id(cell): label for cell, label in zip(umap_cells, umap_labels)}
                    else:
                        umap_label_dict = {}
                else:
                    umap_label_dict = {}

                # 构建新的 cell_list，包含聚类结果
                for i, cell in enumerate(valid_cells):
                    new_cell = dict(cell)
                    cell_name = new_cell['cell_name']
                    cell_name = cell_name.split('_')
                    new_cell['slice_raw'] = int(cell_name[1])
                    new_cell['pca_cluster_label'] = int(pca_labels[i])
                    new_cell['model_cluster_label'] = int(model_labels[i])
                    if id(cell) in umap_label_dict:
                        new_cell['umap_cluster_label'] = int(umap_label_dict[id(cell)])
                    new_cell_list.append(new_cell)

        except Exception as e:
            print(f"{timepoint} 出现错误：{e}")
            print(timepoint)
        target_name = os.path.join("../3D_datasetSplit", f"3Ddataset_timepoint_{timepoint}_cluster.pkl")
        with open( target_name, 'wb') as file:
            pickle.dump(new_cell_list, file)
        print(f"Timepoint {timepoint} 聚类完成并保存到 {target_name}")


def evaluate_clustering(cell_list, cluster_keys, timepoint):
    evaluation_results = {}
    for key in cluster_keys:
        if key == 'umap_cluster_label' and timepoint == 'WT':
            continue

        # 获取聚类标签
        labels = [cell[key] for cell in cell_list if key in cell]
        embeddings = [cell['pca_embeddings'][:64] for cell in cell_list if key in cell]  # PCA 聚类
        embeddings_array = np.stack(embeddings)

        # 计算轮廓系数
        if len(set(labels)) > 1:  # 至少有两个不同的类别才能计算轮廓系数
            silhouette_avg = silhouette_score(embeddings_array, labels)
        else:
            silhouette_avg = -1  # 如果只有一个类别，轮廓系数无法计算，返回 -1

        evaluation_results[key] = silhouette_avg
        print(f"聚类效果评估 ({key}): {silhouette_avg}")

    return evaluation_results
def Cluster_makePng():
    timepoints = ['0hpa1', '12hpa1', '36hpa1', '3dpa1', '5dpa1', '10dpa1', 'WT']
    cluster_keys = ['pca_cluster_label', 'model_cluster_label', 'umap_cluster_label']
    save_dir = "../models/model_3D/3D_cluster_visualizations"
    os.makedirs(save_dir, exist_ok=True)
    slice_num = defaultdict(list)
    for timepoint in timepoints:
        print(f"处理 {timepoint}...")
        target_name = os.path.join("../3D_datasetSplit", f"3Ddataset_timepoint_{timepoint}_cluster.pkl")
        with open(target_name, 'rb') as file:
            cell_list = pickle.load(file)

        # 计算该时间点下所有细胞的全局坐标范围（用于统一图像比例）
        all_positions = [cell['pos2'][:2] for cell in cell_list if 'pos2' in cell]
        all_positions = np.array(all_positions)
        xmin, ymin = np.min(all_positions, axis=0)
        xmax, ymax = np.max(all_positions, axis=0)

        # 按切片分组
        slice_dict = defaultdict(list)
        slice_num = []
        if(timepoint != 'WT'):
            for cell in cell_list:
                if 'slice_raw' in cell:
                    slice_dict[cell['slice_raw']//3].append(cell)
                    slice_num.append(cell['slice_raw'])
        else:
            print(timepoint)
            for cell in cell_list:
                if 'slice_raw' in cell:
                    slice_dict[cell['slice_raw']//3].append(cell)
                    slice_num.append(cell['slice_raw'])
        print(timepoint,Counter(slice_num))
        # 去除细胞数量过少的切片，比如少于 300 个细胞的
        min_cells = 100
        slice_dict = {sid: cells for sid, cells in slice_dict.items() if len(cells) >= min_cells}
        # colormap
        cluster_color_maps = {}
        for key in cluster_keys:
            if key == 'umap_cluster_label' and timepoint == 'WT':
                continue
            labels = [cell[key] for cell in cell_list if key in cell]
            if labels:
                unique_labels = sorted(set(labels))
                cluster_color_maps[key] = {
                    label: plt.get_cmap('tab10')(i % 10) for i, label in enumerate(unique_labels)
                }

        for slice_id, slice_cells in slice_dict.items():
            for key in cluster_keys:
                if key == 'umap_cluster_label' and timepoint == 'WT':
                    continue

                filtered = [cell for cell in slice_cells if key in cell ]
                if not filtered:
                    continue

                positions = np.array([cell['pos2'][:2] for cell in filtered])
                labels = [cell[key] for cell in filtered]
                colors = [cluster_color_maps[key][label] for label in labels]

                plt.figure(figsize=(5, 5))
                plt.scatter(positions[:, 0], positions[:, 1], c=colors, s=0.8, alpha=0.8)
                plt.title(f"{timepoint} | Slice {slice_id} | {key}")
                plt.axis('equal')
                plt.xlim(xmin, xmax)
                plt.ylim(ymin, ymax)
                plt.xticks([])
                plt.yticks([])

                tp_dir = os.path.join(save_dir, timepoint)
                os.makedirs(tp_dir, exist_ok=True)
                fig_name = os.path.join(tp_dir, f"slice_raw_{slice_id}_{key}.png")
                plt.savefig(fig_name,dpi = 1000)
                plt.close()
                print(f"保存图像：{fig_name}")

    pass

In [ ]:
genePcaEmbeddings()
makeModelEmbeddings()
Cluster_make()
Cluster_makePng()


In [ ]:
import pyvista as pv
import pandas as pd
import random
import numpy as np
from scipy.spatial.distance import cdist
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from sklearn.cluster import KMeans
import plotly
import pyvista #'SMESG000063577'
gene_ids = ['SMESG000019132', 'SMESG000081615', 'SMESG000075225', 'SMESG000010938', 'SMESG000077295', 'SMESG000021611']
gene_id_eye = ['SMESG000081129','SMESG000025308','SMESG000025307']
# 数据读取，读取当前时态全部细胞数据
data = pd.read_csv('../models/model_3D/3D_cluster_visualizations/WT/All.txt', sep=' ')
data.columns = ['x', 'y', 'z', 'reflectance']
data['reflectance'] = [0.5] *len(data)
# 随机采样1000个点
data_sampled = data.sample(n=500000, random_state=42)
data_genes = []
for gene_id in gene_ids:
    csv_path = "../models/model_3D/3D_cluster_visualizations/WT/" + gene_id +".txt"
    data_gene = pd.read_csv(csv_path, sep=' ')
    data_gene.columns = ['x', 'y', 'z', 'reflectance']
    data_genes.append(data_gene)
eye_genes = []
for gene_id in gene_id_eye:
    csv_path = "../models/model_3D/3D_cluster_visualizations/WT/" + gene_id +".txt"
    data_gene = pd.read_csv(csv_path, sep=' ')
    data_gene.columns = ['x', 'y', 'z', 'reflectance']
    eye_genes.append(data_gene)
# 创建点云对象
points_combined = np.column_stack((data_sampled['x'], data_sampled['y'], data_sampled['z']))
# points_gene = np.column_stack((data_gene['x'], data_gene['y'], data_gene['z']))

cloud_combined = pv.PolyData(points_combined)
# cloud_gene = pv.PolyData(points_gene)

# 添加reflectance数据
cloud_combined.point_data['reflectance'] = data_sampled['reflectance']
# cloud_gene.point_data['reflectance'] = data_gene['reflectance']
cloud_genes = []
for data_gene in data_genes:
    points_gene = np.column_stack((data_gene['x'], data_gene['y'], data_gene['z']))
    cloud_gene = pv.PolyData(points_gene)
    cloud_gene.point_data['reflectance'] = data_gene['reflectance']
    cloud_genes.append(cloud_gene)
cloud_genes_eye = []
for data_gene in eye_genes:
    points_gene = np.column_stack((data_gene['x'], data_gene['y'], data_gene['z']))
    cloud_gene = pv.PolyData(points_gene)
    cloud_gene.point_data['reflectance'] = data_gene['reflectance']
    cloud_genes_eye.append(cloud_gene)

colors = ["#AA0000",  # 鲜红色
            "#FFA500",  # 橙色
            "#FFFF00",  # 黄色
            "#00FF00",  # 绿色
            "#0000FF",  # 蓝色
            "#800080"]   # 紫色
def get_eye_coor():
    gene_id = 'SMESG000081129'
    # 读取数据
    csv_path = "../models/model_3D/3D_cluster_visualizations/WT/" + gene_id + ".txt"
    data_gene = pd.read_csv(csv_path, sep=' ')

    # 修改列名
    data_gene.columns = ['x', 'y', 'z', 'reflectance']

    # 提取位置坐标
    coordinates = data_gene[['x', 'y', 'z']].values

    # 计算所有点之间的欧氏距离
    distances = cdist(coordinates, coordinates)

    # 计算每个点与其他点的平均距离
    mean_distances = np.mean(distances, axis=1)

    # 设置一个阈值，用于去除那些距离较远的点（可根据数据调整阈值）
    threshold = np.percentile(mean_distances, 60)  # 95%分位数作为阈值，表示去掉最远的5%点

    # 过滤出距离较近的点
    filtered_data = data_gene[mean_distances <= threshold]
    data_gene = filtered_data
    return data_gene
    pass
# 获取过滤后的数据
data_gene = get_eye_coor()
eyes_gene = np.column_stack((data_gene['x'], data_gene['y'], data_gene['z']))

# Step 1: 使用K-means聚类找到两个聚类中心
kmeans = KMeans(n_clusters=2, random_state=42).fit(eyes_gene)
centroids = kmeans.cluster_centers_


# Step 2: 计算圆柱形区域内的点
def generate_cylinder_points(center, R, n):
    """
    根据圆柱体的中心、半径 R 和厚度 n 生成点。
    center: 聚类中心 (x, y, z)
    R: 圆柱的半径
    n: 生成的点的数量
    """
    points = []

    # 生成厚度范围内的点
    for i in range(n):
        # 随机生成 x 和 y 坐标，确保它们在半径 R 内
        angle = random.uniform(0, 2 * np.pi)
        radius = random.uniform(0, R)

        # 计算圆柱表面的坐标
        x = center[0] + radius * np.cos(angle)
        y = center[1] + radius * np.sin(angle)
        z = center[2] + random.uniform(-5, 5)  # 添加厚度的随机z值

        points.append([x, y, z])

    return np.array(points)


# 设置半径 R 和厚度 n
R = 80 # 你可以根据需要调整这个值
n = 1000  # 生成的点数（厚度）

# Step 3: 对每个聚类中心生成圆柱形区域内的点
all_new_points = []
eye_new_cells = []
# print(centroids)
for centroid in centroids:
    new_points = generate_cylinder_points(centroid, R, n)
    all_new_points.append(new_points)

new_points_item = generate_cylinder_points(np.array([230,-80,16.1875]), int(R/5), int(n/5))
eye_new_cells.append(new_points_item)
new_points_item = generate_cylinder_points(np.array([210,30 ,22.1875]), int(R/5), int(n/5))
eye_new_cells.append(new_points_item)
# 合并原始点和新点
new_eyes_gene = np.vstack([eyes_gene, *all_new_points])
eye_new_cells = np.vstack([*eye_new_cells])
for i in range(0,2):
    print(i)
    # 创建可视化窗口
    plotter = pv.Plotter(off_screen=True) #off_screen=True
    # 创建 `cloud_combined` 的外壳，包围盒或最小外接球
    # 在这里，我们创建一个包围盒作为外壳，可以根据实际需求选择球体或者其他形状
    #bounding_box1 = cloud_combined.bounds  # 获取包围盒的边界
    bounding_box =  (-16.0, 4306.0, -548.0, 567.0, -250, 250.0) #  (-16.0, 4306.0, -548.0, 567.0, -155.0, 200.0)

    shell = pv.Cube(center=((bounding_box[0] + bounding_box[1]) / 2,
                            (bounding_box[2] + bounding_box[3]) / 2,
                            (bounding_box[4] + bounding_box[5]) / 2),
                    x_length=bounding_box[1] - bounding_box[0],
                    y_length=bounding_box[3] - bounding_box[2],
                    z_length=bounding_box[5] - bounding_box[4])
    # 提取外壳的边框
    shell_edges = shell.extract_feature_edges()
    # 外壳：淡蓝色，提高层次感
    plotter.add_mesh(shell, color='lightsteelblue', opacity=0.25)
    #plotter.add_mesh(shell_edges, color='black', line_width=10)
    # 点云前景：使用淡橙色，更加清晰且有活力
    plotter.add_points(cloud_combined, color='lightsteelblue', opacity=0.5, render_points_as_spheres=True, point_size=10)
    # 基因点：使用鲜亮的绿色，使其在背景上更突出'darkviolet'
    plotter.add_points(cloud_genes[i], color='darkviolet', opacity=1, render_points_as_spheres=True, point_size=30)
    # 补充点：柔和的天蓝色，使其与整体结构融合
    plotter.add_points(new_eyes_gene, color='white', opacity=0.5, render_points_as_spheres=True, point_size=20)
    # 眼部新细胞：深紫色，比黑色更柔和，但仍然突出
    plotter.add_points(eye_new_cells, color='black', opacity=0.5, render_points_as_spheres=True, point_size=25)
    # # plotter.reset_camera()
    # 默认相机位置: [(2145.0, 9.5, 8672.565914523146),
    #  (2145.0, 9.5, 22.5),
    #  (0.0, 1.0, 0.0)]
    shell_center = (
        (bounding_box[0] + bounding_box[1]) / 2,  # x_center
        (bounding_box[2] + bounding_box[3]) / 2,  # y_center
        (bounding_box[4] + bounding_box[5]) / 2  # z_center
    )
    print(bounding_box) # (-16.0, 4306.0, -548.0, 567.0, -155.0, 200.0)

    shell_size = max(
        bounding_box[1] - bounding_box[0],
        bounding_box[3] - bounding_box[2],
        bounding_box[5] - bounding_box[4]
    )

    # camera_position = [
    #     (shell_center[0] , -2500, shell_center[2]+ 1.5* shell_size),  # 相机位置
    #     shell_center,  # 焦点
    #     (0, 1, 0)  # Z 轴朝上
    # ]
    camera_position = [(2145.0, -6500, 6505.5), (2145.0, 9.5, 22.5), (0, 1, 0)]
    # print("默认相机位置:",camera_position) # [(2145.0, -2500, 6505.5), (2145.0, 9.5, 22.5), (0, 1, 0)]
    # # 设置相机位置
    plotter.camera_position = camera_position
    # plotter.screenshot(gene_ids[i] + '_WT.png', window_size=[8000, 6000])
    plotter.screenshot('./snapshot3/'+gene_ids[i]+'_WT.png', window_size=[8000, 6000])



In [ ]:
from sklearn.metrics import davies_bouldin_score, calinski_harabasz_score


def makeUMAPbyEmbeddings():
    # 根据pca或pfm的embedding进行umap降维并展示
    timepoints = ['0hpa1', '12hpa1', '36hpa1', '3dpa1', '5dpa1', '10dpa1', 'WT']
    for timepoint in timepoints:
        print(f"处理 {timepoint}...")
        target_name = os.path.join("./3D_datasetSplit", f"3Ddataset_timepoint_{timepoint}_cluster.pkl")
        with open(target_name, 'rb') as file:
            cell_list = pickle.load(file)
        pfm_embeddings = np.stack([cell['model_embeddings'][:-1].cpu().numpy() for cell in cell_list])
        pca_model_low = PCA(n_components=64)
        pfm_embeddings = pca_model_low.fit_transform(pfm_embeddings)
        pca_embeddings = np.array([cell['pca_embeddings'][:64] for cell in cell_list])
        pfm_clusters = np.array([cell['model_cluster_label'] for cell in cell_list])
        pca_clusters = np.array([cell['pca_cluster_label'] for cell in cell_list])
        print('umapping ...')
        # UMAP降维：从64维降成2维
        reducer = umap.UMAP(n_neighbors=500, min_dist=0.1, metric='cosine')
        pfm_umap = reducer.fit_transform(pfm_embeddings)
        pca_umap = reducer.fit_transform(pca_embeddings)
        pfm_socre_db = davies_bouldin_score(pfm_umap, pfm_clusters)
        pca_socre_db = davies_bouldin_score(pca_umap, pca_clusters)
        pfm_score_ch = calinski_harabasz_score(pfm_umap, pfm_clusters)
        pca_score_ch = calinski_harabasz_score(pca_umap, pca_clusters)
        print(f'{timepoint} pca_score_ch: {pca_score_ch} pfm_score_ch: {pfm_score_ch} pca_score_db: {pca_socre_db} pfm_socre_db: {pfm_socre_db}')
        # 0hpa1 pca_score_ch: 88573.25859832646 pfm_score_ch: 156127.28570893584 pca_score_db: 9.572714934063447 pfm_socre_db: 1.5098163138300325
        # 12hpa1 pca_score_ch: 90707.08556523663 pfm_score_ch: 215729.84225361526 pca_score_db: 4.472347586866109 pfm_socre_db: 1.375367608071456
        # 36hpa1 pca_score_ch: 136404.74377893948 pfm_score_ch: 181306.6258071437 pca_score_db: 13.14868620054698 pfm_socre_db: 1.8749485404025727
        # 3dpa1 pca_score_ch: 72253.90564719414 pfm_score_ch: 173945.59900259832 pca_score_db: 20.786527920860827 pfm_socre_db: 1.4828635699994575
        # 5dpa1 pca_score_ch: 89058.55903515818 pfm_score_ch: 108571.26657052618 pca_score_db: 15.097172816700143 pfm_socre_db: 1.9905214842116918
        # 10dpa1 pca_score_ch: 76334.67945281166 pfm_score_ch: 51583.462371823174 pca_score_db: 15.160026120551318 pfm_socre_db: 2.512483721024162
        # WT pca_score_ch: 355290.01226107887 pfm_score_ch: 907662.982877599 pca_score_db: 7.7558976730019875 pfm_socre_db: 1.1598260398130495
        # 绘图函数
        def plot_umap(umap_result, labels, title, filename, color_map):
            plt.figure(figsize=(10, 10))
            colors = [color_map[label] for label in labels]
            plt.scatter(umap_result[:, 0], umap_result[:, 1], c=colors, s=10, alpha=0.8)
            # plt.title(title)
            # plt.xlabel("UMAP-1")
            # plt.ylabel("UMAP-2")
            plt.xticks([])
            plt.yticks([])
            plt.tight_layout()
            plt.savefig(filename, dpi=300)
            plt.close()
        print('make png ...')
        # 保存图像
        os.makedirs("./UMAP_results", exist_ok=True)
        # 创建统一颜色映射（只对 pfm_clusters / pca_clusters）
        cluster_color_maps = {}

        for label_type, labels in [('pfm', pfm_clusters), ('pca', pca_clusters)]:
            unique_labels = sorted(set(labels))
            cluster_color_maps[label_type] = {
                label: plt.get_cmap('tab10')(i % 10) for i, label in enumerate(unique_labels)
            }

        # 生成并保存图
        plot_umap(pfm_umap, pfm_clusters,
                  f"{timepoint} - PFM Embedding",
                  f"./UMAP_results/{timepoint}_pfm_umap.png",
                  cluster_color_maps['pfm'])

        plot_umap(pca_umap, pca_clusters,
                  f"{timepoint} - PCA Embedding",
                  f"./UMAP_results/{timepoint}_pca_umap.png",
                  cluster_color_maps['pca'])

        print(f"{timepoint} 完成，图像已保存。")
    pass
makeUMAPbyEmbeddings()

In [ ]:

import sns


def makeEvalPNG():
# 0hpa1 pca_score_ch: 88573.25859832646 pfm_score_ch: 156127.28570893584 pca_score_db: 9.572714934063447 pfm_socre_db: 1.5098163138300325
# 12hpa1 pca_score_ch: 90707.08556523663 pfm_score_ch: 215729.84225361526 pca_score_db: 4.472347586866109 pfm_socre_db: 1.375367608071456
# 36hpa1 pca_score_ch: 136404.74377893948 pfm_score_ch: 181306.6258071437 pca_score_db: 13.14868620054698 pfm_socre_db: 1.8749485404025727
# 3dpa1 pca_score_ch: 72253.90564719414 pfm_score_ch: 173945.59900259832 pca_score_db: 20.786527920860827 pfm_socre_db: 1.4828635699994575
# 5dpa1 pca_score_ch: 89058.55903515818 pfm_score_ch: 108571.26657052618 pca_score_db: 15.097172816700143 pfm_socre_db: 1.9905214842116918
# 10dpa1 pca_score_ch: 76334.67945281166 pfm_score_ch: 51583.462371823174 pca_score_db: 15.160026120551318 pfm_socre_db: 2.512483721024162
# WT pca_score_ch: 355290.01226107887 pfm_score_ch: 907662.982877599 pca_score_db: 7.7558976730019875 pfm_socre_db: 1.1598260398130495
    # 构建数据
    data = {
        "Timepoint": ["0hpa1", "12hpa1", "36hpa1", "72hpa1", "120hpa1", "240hpa1", "WT"],
        "PFM_ch": [156127.29, 215729.84, 181306.63, 173945.60, 108571.27, 51583.46, 907662.98],
        "PCA_ch": [88573.26, 90707.09, 136404.74, 72253.91, 89058.56, 76334.68, 355290.01],
        "PFM_db": [1.51, 1.38, 1.87, 1.48, 1.99, 2.51, 1.16],
        "PCA_db": [9.57, 4.47, 13.15, 20.79, 15.10, 15.16, 7.76]

    }

    df = pd.DataFrame(data)

    # 将数据转换成长格式便于绘图
    df_ch = df.melt(id_vars="Timepoint", value_vars=["PFM_ch", "PCA_ch"],
                    var_name="Method", value_name="ch_score")
    df_db = df.melt(id_vars="Timepoint", value_vars=["PFM_db", "PCA_db"],
                    var_name="Method", value_name="db_score")

    # 替换方法名
    df_ch["Method"] = df_ch["Method"].str.replace("_ch", "")
    df_db["Method"] = df_db["Method"].str.replace("_db", "")

    # 绘图：ch 分数
    plt.figure(figsize=(10, 8))
    sns.barplot(data=df_ch, x="Timepoint", y="ch_score", hue="Method")
    plt.title("PFM vs PCA on Calinski-Harabasz Score", fontsize = 20)
    plt.xlabel("Timepoint", fontsize=20)
    plt.ylabel("CH Score", fontsize=20)
    plt.xticks(rotation=45, fontsize=20)
    plt.yticks(fontsize=20)
    plt.legend(fontsize=20)
    plt.tight_layout()

    plt.savefig("./UMAP_results/CH.png",dpi = 1000)
    # 绘图：db 分数
    plt.figure(figsize=(10, 8))
    sns.barplot(data=df_db, x="Timepoint", y="db_score", hue="Method")
    plt.title("PFM vs PCA on Davies-Bouldin Score", fontsize = 20)
    plt.xlabel("Timepoint", fontsize=20)
    plt.ylabel("DB Score", fontsize=20)
    plt.xticks(rotation=45, fontsize=20)
    plt.yticks(fontsize=20)
    plt.legend(fontsize=20)
    plt.tight_layout()

    plt.savefig("./UMAP_results/DB.png", dpi=1000)

makeEvalPNG()